In [2]:
import secrets
from operator import truediv
from wsgiref import util

'''

Scenario :
AIS monitors traffic coming in and out of the local network at an IP level and identifies malformed packages

How the AIS works :
Lymphocytes detect patterns not in the self set (= detector set = unusual packages)
When enough of the lymphocytes detectors detect elements of the detector set
Then the lymphocytes activation threshold is reached which triggers an alarm

Plan for gerüst :
1. write lymphocyte class
    1a. create antibody class
        a.set up detects method specifics of detects
            a1. Implement detects based on packages in scapy this is implemented when detects is true when antibody = package
                - decide on maths
                    - determine level of threshhold for detection = stimlation = how similar does antibody have to be to package to trigger alarm
                    - decide what datastructures are we using and how is the comparison performed
                    - in the first version use maths from demo (if it can be used on larger packages)
2. create self set based on normal packages in scapy
    a. create example packets to test that the normal_traffic and antibody connect correctly
    b. download a file (write down details of training data to account for biases later) with example packets (for malicious and not malicious) and use them as example traffic (download, filter for http/https, convert to bits)
3. write the self set (lyphocytes that dont detect normal_packets) using negative selection

Future Improvements :
-> create a virtual environment to send the packets and lympocytes through using mininet so that lymphocites are trained in real time
    -> change the packages/traffic based on the latest threats?
-> add lymphocites dying like the architecture.. paper and being replaced - a population of lyphocytes
-> add memory detectors (and limit them) like the architecture .. paper
-> model primary response (lymphocytes that have detected a new pathogen multiply because the others dont have the antibodies to recognise it)  and secondary response based on this paper https://www.researchgate.net/profile/Steven-Hofmeyr/publication/12197865_Architecture_for_an_Artificial_Immune_System/links/00b7d538f8550ca681000000/Architecture-for-an-Artificial-Immune-System.pdf (generally do it like that paper to improve)
-> make it so sensitivity level mimics cytokines (signaling other lymphocytes) if there were already activations in the region?
-> Have ABNORMAL_HTTP_TRAFFIC simulate not just malformed packages but also malicious packages that follow the rules of the protocol packets but simulate : IPV4 packet fragmentation attacks, TCP Null scan, TCP FIN scan, TCP XMas Scan, TCP flag manipulation, TCP sequence number anomalies, ICMP based attacks, DNS tunneling, DNS poisoning, DHCP spoofing, protocol fuzzing
-> define the threshold of activation not for once lyphocyte but as a group so that - for example - it the AIS is deployed across several vlans of a company network their joint activation thresshold enables us to recognise attack patterns across several
vlans (that use different firwalls and siems so that the detection of overall patterns is an isusue).
-> instead of making the generated antibodies random make them more like possible malicious packages without narrowing down the detector coverage

Distant Future Improvements
-> work out your own problem representation for how antigens are constructed (see review paper https://staff.fmi.uvt.ro/~daniela.zaharie/am2016/proiecte/tehnici/AIS/AIS_advances.pdf)
-> use flow records instead of raw packages (so the relationship between the packages can be considered)
-> inspect packets at TCP and DNS level
'''

'\nBasic Premise\nLymphocytes detect patterns not in the self set (= detector set = unusual packages)\nWhen enough of the lymphocytes detectors detect elements of the detector set\nThen the lymphocytes activation threshold is reached which triggers an alarm\n\nPlan for gerüst\n1. write lymphocyte class\n    1a. create antibody class\n        a.set up detects method specifics of detects left vague because we dont know how to process the packages provided by this library effiently\n2. create self set based on normal packages in scapy\n3. write the lyphocyte set (lyphocytes that dont detect packages in the self set) using negative selection\n'

In [16]:
import random
from scapy.layers.inet import IP, TCP
from scapy.packet import Packet
from enum import Enum, auto
from bitarray import bitarray as BitArray
import numpy as np
"this cell contains network related classes"

"this class creates packages that simulate a normal https connection a list of typical worksites is used "
#TODO : make it https instead of http


packets_in_collection = 5 #columns
bits_in_packet = 5 #rows
Normal_Packet_Collection = np.empty((packets_in_collection, bits_in_packet), dtype = np.bool_)
Abnormal_Packet_Collection = np.empty((packets_in_collection, bits_in_packet), dtype = np.bool_)
Mixed_Packet_Collection = np.empty((packets_in_collection, bits_in_packet), dtype = np.bool_)

"used to identify duplicate lymphocytes"
Lymphocyte_Set = None


"Defines the type of traffic"
class PacketCollectionType(Enum):
        NORMAL = auto()
        ABNORMAL = auto()
        MIXED = auto()

packet_collections = {
    PacketCollectionType.NORMAL : Normal_Packet_Collection,
    PacketCollectionType.ABNORMAL : Abnormal_Packet_Collection,
    PacketCollectionType.MIXED : Mixed_Packet_Collection
}


"""This function creates a list representing the self set. Any detector the detect method determines too close to an element of the self set is deleted"""
def create_packet_set(packet_collection : PacketCollectionType,src_IP, dst_IP,dport):
        #TODO: call a method that creates the packages and save them as a WHAT?
        #only save/use (source host IP address,destination host IP address,TCP service/port number) LISYS
        if packet_collection == PacketCollectionType.NORMAL:
            self_set : list[Packet] = []
            while len(self_set) < packet_collection_size:
                packet = IP(src=random.choice(src_IP),dst = random.choice(dst_IP))/TCP(sport = random.choice(dport))
                self_set.append(packet)
                #check that self set is a list of valid packages
                #convert self_set to binary
            return self_set
        elif packet_collection == PacketCollectionType.ABNORMAL:
            pass
        elif packet_collection == PacketCollectionType.MIXED:
            pass

#def convert_collection_to_bits(packet_collection : list[list]) ->  packet_collections[PacketCollectionType] :
    #list_number = 0
    #element_number = 0

    #step 1 turn each value into a 2D numpy array
    #while packet_collection[list_number] < len(packet_collection):
        #ip_field = packet_collection[list_number[element_number]]
        #if type(ip_field) == int :
            #sport = bin(ip_field,16)
        #elif type(ip_field) == str and element_number == 0 :
            #src =
    #step 2 after conversion each array is added to a np matrix to create one packet
    #step 3 the packet is added to





class Normal_HTTP_Traffic:

        """This function converts the list of packages to a list of binaries"""
        def self_set_to_binary(self):
            pass

        def __init__(self):

            self.src_IP : list[str] = ["192.168.1.1", "192.168.1.2", "192.168.1.3", "192.168.1.4", "192.168.1.5"]
            self.dst_IP : list[str] = ["192.168.1.10", "192.168.1.20", "192.168.1.30", "192.168.1.40", "192.168.1.50"]
            self.sport : list[int] = [49152]
            #use this to test different values in the packet fields and using actual packets
            #self.self_set : list[Packet] = create_packet_set(PacketCollectionType.NORMAL,self.src_IP,self.dst_IP,self.sport)
            #use this to test if the packet list can be connected to the bit converter and the rest of the program
            self.dummy_self_set : list[list] = [["192.168.1.1","192.168.1.10", 49152],["192.168.1.2","192.168.1.20", 49152]]

        def __repr__(self) -> str:
            return f"{type(self).__name__}(self set = {self.dummy_self_set})"

"Creates malformed packages by fuzzing them and creates list of normal and abnormal packets"
class Abnormal_Traffic:
    pass



#self_set_object = Normal_HTTP_Traffic()
#firstpackage =  self_set_object.self_set[0]
"in scapy you access layers by class"
src_value = firstpackage[IP].src
print(type(src_value))
print(src_value)
#repr(self_set_object)



<class 'scapy.base_classes._ScopedIP'>
192.168.1.3


In [10]:
import bitarray
from bitarray import bitarray as BitArray
import bitarray.util
import pytest

"AIS classes : antibody and lymphocyte are defined below"

class Antibody:
    #TODO put a __repr__ method here
    def __new__(cls, *args,**kwargs):
        print("Created a new instance of antibody.")
        return super().__new__(cls)

    def __init__(self):
        #TODO : research what can go wrong with bits as a datatype and prevent
        # idea print(type(self.detector))
        self.detector : BitArray = bitarray.util.random_k(4,2,endian = "big")
        #TODO : write test that runs when an antibody is created that tests if its identical to itself? valid?

    "the logic of this function compares the new antibody with the current self set using r contiguous matching the higher r is the more detectors needed for usefull detection """
    def detects(self, Normal_Packet_Collection) -> bool:
        # use r contiguous bits to determine similarity
        # the datapath triple is being analysed
        #TODO Implement r contiguous bits
        pass

class Lymphocyte:
    def __new__(cls, *args,**kwargs):
        print("Created a new instance of Lymphocyte.")
        #super allows access to parent class object we give its constructor
        #lymphocyte as an argument
        #we use a double underscore because line 7 calls a special method
        return super().__new__(cls)
    #self holds a reference to the current instance
    def __init__(self,antibody,stimulation):
        #TODO: change single antibody an antibody array
        self.antibody = Antibody()
        #TODO : implement stimulation
        self.stimulation = stimulation
    def __repr__(self) -> str:
        return f"{type(self).__name__}(antibody = {self.antibody.detector}, stimulation = {self.stimulation})"

    """activation threshold of lymphocyte is exceeded when lymphocyte detects X number of antigens in a short period of time"""
    def is_threshold_exceeded(self):
        pass

    """"If lymphocyte is not unique it is discarded"""
    def is_lymphocyte_unique(self):
        pass


lymphocyte = Lymphocyte("antifabody", "no stimulation")
lymphocyte.antibody.detects(lymphocyte.antibody.detector)
repr(lymphocyte)

Created a new instance of Lymphocyte.
Created a new instance of antibody.


"Lymphocyte(antibody = bitarray('1001'), stimulation = no stimulation)"